<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">Faculty AI Seminar</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 2: Build a Discipline-Specific AI Teaching Assistant</h2>
</div>

## What you'll build today

By the end of this 90-minute lab, you will have a working AI tutor that:

1. **Knows your discipline's content** — grounded in a real document from your field
2. **Cites its sources** — no hallucinations, no made-up facts
3. **Generates ready-to-use teaching artifacts** — quiz questions, study guides, or grading rubrics
4. **Travels with you** — download the notebook, run it with new documents anytime

## How to read this notebook

- **Gray boxes** are notes — read them.
- **White boxes with `▶`** are code — click in and press `Shift + Enter` to run them.
- **Cells marked 🟢 EDIT ME** are the only ones you need to change. Everything else just runs.

Work top-to-bottom. Don't skip cells — some depend on earlier ones.

---
# Part 1 — Setup *(watch the instructor for this part)*

We need to install some tools and confirm we have access to the AI models. The instructor will walk through these cells — you don't need to touch them yet.

### 1.1 Install dependencies

This installs the Python libraries we'll use. Takes about 30 seconds.

In [ ]:
%%capture
!pip install -q -r requirements.txt

### 1.2 Import the tools we need

Think of this like setting tools out on a workbench before starting a project.

In [ ]:
import boto3
import json
import warnings
from typing import List
from IPython.display import Markdown, display

from langchain_aws import ChatBedrockConverse
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

warnings.filterwarnings("ignore")
print("Imports ready.")

### 1.3 Connect to Amazon Bedrock

Bedrock is AWS's service for accessing foundation models like Nova, Claude, and Llama. We connect once and then call models whenever we want.

In [ ]:
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
print("Connected to Bedrock.")

---
# Part 2 — Talk to a model directly *(your turn)*

PartyRock hid the code. Here it is. A model call is just three lines: define a prompt, pick a model, get a response.

### 2.1 Set up two models so we can compare them

We'll use **Nova Lite** (fast, frugal — AWS's lightweight model) and **Claude Haiku** (Anthropic's fast model). Same prompt, different voices.

In [ ]:
nova_lite = ChatBedrockConverse(
    model="amazon.nova-lite-v1:0",
    temperature=0.7,
    max_tokens=500,
)

claude_haiku = ChatBedrockConverse(
    model="anthropic.claude-3-haiku-20240307-v1:0",
    temperature=0.7,
    max_tokens=500,
)

print("Both models ready.")

### 2.2 🟢 EDIT ME — Try a prompt

Change the prompt below to anything you'd ask a colleague or student. Then run it. **Try a topic from your discipline.**

In [ ]:
prompt = "Write a 4-line haiku about teaching in my field. My field is: [your field here]"

response = nova_lite.invoke(prompt)
display(Markdown(f"**Nova Lite says:**\n\n{response.content}"))

### 2.3 Same prompt, different model

Run the cell below to see how Claude Haiku answers the same prompt. Notice the difference in voice and style.

In [ ]:
response = claude_haiku.invoke(prompt)
display(Markdown(f"**Claude Haiku says:**\n\n{response.content}"))

> 💡 **Why this matters:** PartyRock picks a model for you. In code, *you* pick. Different models cost different amounts, have different strengths, and produce different voices. This is the first thing you outgrow PartyRock for.

---
# Part 3 — Ground the AI in YOUR discipline's content

This is the part where things get useful. We're going to load a real document from your field and have the AI answer questions using **only** that document. No hallucinations. No invented facts. Every answer is traceable to the source.

This pattern is called **Retrieval-Augmented Generation (RAG)** — fancy name for "give the model the textbook before you ask it the question."

### 3.1 🟢 EDIT ME — Pick your document

We've pre-loaded a sample document for each persona. **Change the path below to match your persona** (or use the one already set — it's been matched to you at registration).

| Persona | File path |
|---|---|
| 1. Dentistry | `data/persona1_dentistry_perio_case.pdf` |
| 2. Computer Science | `data/persona2_cs_data_structures.pdf` |
| 3. English Literature | `data/persona3_english_victorian_essay.pdf` |
| 4. Nursing | `data/persona4_nursing_pharmacology.pdf` |
| 5. Business | `data/persona5_business_leadership_case.pdf` |
| 6. Biology | `data/persona6_biology_lab_protocol.pdf` |

*Optional:* If you brought your own PDF (under ~50 pages), drag it into the `data/` folder in the file browser on the left, then update the path here.

In [ ]:
pdf_path = "data/persona1_dentistry_perio_case.pdf"

### 3.2 Load the document

Just press run. This reads the PDF page by page.

In [ ]:
pages = PyPDFLoader(pdf_path).load()
print(f"Loaded {len(pages)} pages from {pdf_path}")

display(Markdown(f"**Preview of page 1:**\n\n{pages[0].page_content[:800]}..."))

### 3.3 Chunk the document into searchable pieces

AI models have limited "working memory." We can't fit a 25-page PDF into a single prompt, so we split it into overlapping chunks. The AI will later pull the most relevant chunks when asked a question.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=60,
    separators=["\n\n", "\n", "(?<=\\. )", " ", ""],
    is_separator_regex=True,
)

chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

### 3.4 Convert chunks to embeddings and index them

An *embedding* is a numeric fingerprint of a piece of text. Chunks with similar meaning have similar fingerprints, so when you ask a question, we can quickly find the most relevant chunks. This is the search engine your AI tutor will use.

*This cell takes 30–60 seconds to run — it's calling Bedrock once per chunk.*

In [ ]:
class NovaMultimodalEmbeddings(Embeddings):
    """Embeddings using Amazon Nova 2 Multimodal Embeddings."""

    def __init__(self, client, model_id="amazon.nova-2-multimodal-embeddings-v1:0", dimension=1024):
        self.client = client
        self.model_id = model_id
        self.dimension = dimension

    def _embed(self, text: str, purpose: str) -> List[float]:
        body = {
            "taskType": "SINGLE_EMBEDDING",
            "singleEmbeddingParams": {
                "embeddingDimension": self.dimension,
                "embeddingPurpose": purpose,
                "text": {"truncationMode": "END", "value": text},
            },
        }
        response = self.client.invoke_model(modelId=self.model_id, body=json.dumps(body))
        result = json.loads(response["body"].read())
        return result["embeddings"][0]["embedding"]

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(t, "GENERIC_INDEX") for t in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text, "GENERIC_RETRIEVAL")


bedrock_embeddings = NovaMultimodalEmbeddings(client=bedrock_runtime)

print("Building the vector index... (this takes 30–60 seconds)")
vectordb = FAISS.from_documents(chunks, bedrock_embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print("Index ready.")

### 3.5 Build the Q&A chain

Now we wire it all together. The chain: question → retrieve relevant chunks → ask the model using ONLY those chunks → return the answer.

In [ ]:
qa_template = """You are a discipline-specific teaching assistant. Use ONLY the context below to answer the question.
If the context doesn't contain the answer, say "I don't have that information in the source material." Do not make things up.

Context:
{context}

Question: {question}

Answer (cite specific details from the context):
"""

qa_prompt = PromptTemplate.from_template(qa_template)
qa_chain = qa_prompt | nova_lite | StrOutputParser()

def ask(question: str):
    """Ask a question grounded in the loaded document."""
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    answer = qa_chain.invoke({"question": question, "context": context})
    return answer, docs

print("Q&A chain ready. Try it below.")

### 3.6 🟢 EDIT ME — Ask your document a question

In [ ]:
question = "What are the key concepts a student should learn from this material?"

answer, sources = ask(question)
display(Markdown(f"### Answer\n\n{answer}"))

print(f"\nAnswer drawn from {len(sources)} source chunks:")
for i, doc in enumerate(sources, 1):
    page = doc.metadata.get("page", "?")
    print(f"  [{i}] page {page}: {doc.page_content[:120]}...")

### 3.7 Compare: grounded answer vs. ungrounded answer

This is the moment skeptics convert. Same question, two answers:

- **Left:** What the AI says when grounded in YOUR document
- **Right:** What the AI says when answering from its general training

Watch what happens with a discipline-specific question.

In [ ]:
comparison_question = question  # uses your question from 3.6

grounded_answer, _ = ask(comparison_question)
vanilla_answer = nova_lite.invoke(comparison_question).content

display(Markdown(f"### 🟢 Grounded in YOUR document\n\n{grounded_answer}\n\n---\n\n### 🔴 Ungrounded (AI guessing from training)\n\n{vanilla_answer}"))

> 💡 **What you should see:** The grounded answer sticks to facts from your document. The ungrounded answer may sound confident but invent details, drift off-topic, or contradict your source. **This is why faculty should care.**

---
# Part 4 — Generate a teaching artifact

Three templates below. Pick the one most useful for your course (or run all three). Each one will use your loaded document to produce something you could hand to a TA, a student, or a colleague.

### Template A — Quiz Generator

In [ ]:
quiz_request = """Based on the source material, generate 5 quiz questions for undergraduate students.
Mix 3 multiple-choice questions (with 4 options each) and 2 short-answer questions.
Include an answer key. For each question, cite which section or page of the source it tests.
Format as clean markdown."""

quiz_output, _ = ask(quiz_request)
display(Markdown(quiz_output))

### Template B — Study Guide

In [ ]:
study_guide_request = """Create a one-page study guide from the source material with three sections:

1. **Key Concepts** — 5 concepts, each with a one-sentence definition
2. **Flashcards** — 10 term/definition pairs in a table
3. **Likely Exam Questions** — 3 questions a student should be able to answer

Use clean markdown formatting."""

study_guide_output, _ = ask(study_guide_request)
display(Markdown(study_guide_output))

### Template C — Grading Rubric

In [ ]:
rubric_request = """Draft a 4-level grading rubric (Excellent / Proficient / Developing / Beginning)
for assessing student understanding of the source material.
Include 4 criteria: factual accuracy, application to new contexts, critical analysis, and use of source evidence.
Format as a markdown table."""

rubric_output, _ = ask(rubric_request)
display(Markdown(rubric_output))

### 4.4 Save your artifact to take home

This writes the outputs to a markdown file you can download. **Right-click the file in the file browser on the left → Download.**

In [ ]:
from datetime import datetime

filename = f"my_ai_teaching_artifact_{datetime.now().strftime('%Y%m%d_%H%M')}.md"

with open(filename, "w") as f:
    f.write(f"# AI Teaching Artifacts\n\n")
    f.write(f"Source document: `{pdf_path}`\n\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n---\n\n")
    f.write(f"## Quiz\n\n{quiz_output}\n\n---\n\n")
    f.write(f"## Study Guide\n\n{study_guide_output}\n\n---\n\n")
    f.write(f"## Rubric\n\n{rubric_output}\n")

print(f"Saved to: {filename}")
print("Right-click it in the file browser and choose 'Download'.")

---
# Part 5 — Share with your pod

Pick **one** output you generated. Paste it into the seminar Slack channel under the thread for your persona.

When you share, include:
1. **Your discipline**
2. **Which template you used** (quiz / study guide / rubric)
3. **One thing that surprised you** — good or bad

We'll spotlight 2–3 in the wrap-up at 3:45.

---
## What to try next (Monday morning)

Now that you've built one, here's what you could change to make it your own:

1. **Swap the document** — change `pdf_path` to your actual syllabus, lecture notes, or textbook chapter. Re-run from Part 3.
2. **Edit the prompt templates** — make the quiz match your testing style. Make the rubric match your department's standards.
3. **Tighten the grounding** — add `"Refuse to answer if the context is unclear"` to the QA template if you're worried about edge cases.
4. **Add a second document** — load multiple PDFs and combine them into one vector store. Now your AI tutor knows your whole course.
5. **Try a different model** — change `nova_lite` to `claude_haiku` in the chain. Compare outputs.

### Where to go deeper

The full MLU EEP Generative AI curriculum has 14 lessons and 13 labs covering responsible AI, agents, and multimodal applications:

- [aws-samples/aws-mlu-eep-generative-ai](https://github.com/aws-samples/aws-mlu-eep-generative-ai) — full source curriculum
- **Module 3 Lab 3a — RAG deep dive** (this lab is built on its patterns)
- **Module 3 Lab 4 — Agents** (the next step beyond Q&A)

Your persistent Slack channel from today's seminar is your follow-up home for questions.